In [ ]:
import asyncio
import websockets
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyeeg as pe
import warnings
import os
import pickle
import nest_asyncio
from collections import deque
import threading
import http.server
import socketserver

nest_asyncio.apply()
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Resolve project root (works from notebooks_deap/ folder)
_cwd = os.path.abspath(os.getcwd())
_candidates = [_cwd] + [os.path.abspath(os.path.join(_cwd, *([".."]*i))) for i in range(1, 6)]
PROJECT_ROOT = next((p for p in _candidates if os.path.isdir(os.path.join(p, "data", "DEAP", "data_preprocessed_python"))), _cwd)

BASE_PATH = os.path.join(PROJECT_ROOT, "data", "DEAP", "output", "work")
MODEL_PATH = os.path.join(BASE_PATH, 'best_model.pth')
SCALER_PATH = os.path.join(BASE_PATH, 'scaler.pkl')

LOCAL_PORT = 65432       # WebSocket port for streamer communication
HTTP_PORT = 8000         # HTTP port for the web dashboard

N_CH = 16
N_FEATS = 10
WINDOW_SIZE = 256        # 2 seconds at 128Hz
INFERENCE_STEP = 32      # Run inference every 32 new samples (~0.25s)
BUFFER_CAPACITY = int(5 / (INFERENCE_STEP / 128))  # ~5s smoothing window

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_PATH:   {MODEL_PATH}")
print(f"SCALER_PATH:  {SCALER_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
class GATLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, num_heads: int = 4,
                 attn_dropout: float = 0.0, residual: bool = True):
        super().__init__()
        self.H = num_heads
        self.d = out_features
        self.residual = residual
        self.W = nn.Linear(in_features, num_heads * out_features, bias=False)
        self.a = nn.Parameter(torch.empty(num_heads, 2 * out_features))
        nn.init.xavier_uniform_(self.a.unsqueeze(0))
        self.leaky = nn.LeakyReLU(0.2)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.bn = nn.BatchNorm1d(out_features)
        if residual:
            self.res_proj = (nn.Linear(in_features, out_features, bias=False)
                             if in_features != out_features else nn.Identity())

    def forward(self, x: torch.Tensor):
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.H, self.d)
        hi = h.unsqueeze(2)
        hj = h.unsqueeze(1)
        pair = torch.cat([hi.expand(B, N, N, self.H, self.d),
                          hj.expand(B, N, N, self.H, self.d)], dim=-1)
        e = self.leaky((pair * self.a.unsqueeze(0).unsqueeze(0).unsqueeze(0)).sum(-1))
        alpha = self.attn_drop(F.softmax(e, dim=2))
        out = torch.einsum("bqkh, bkhd -> bqhd", alpha, h).mean(dim=2)
        out = self.bn(out.reshape(B * N, self.d)).reshape(B, N, self.d)
        out = F.elu(out)
        if self.residual:
            out = out + self.res_proj(x)
        return out, alpha.permute(0, 3, 1, 2)


class TaskHead(nn.Module):
    def __init__(self, in_dim: int, dense: int, head_dropout: float = 0.0):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, dense), nn.LayerNorm(dense), nn.GELU(),
            nn.Dropout(head_dropout), nn.Linear(dense, 1),
        )
    def forward(self, x):
        return self.head(x.mean(dim=1))


class DeepGAT(nn.Module):
    def __init__(self, n_channels: int, in_feats: int, backbone_dims: list,
                 dense_size: int, num_heads: int = 4, attn_dropout: float = 0.0,
                 head_dropout: float = 0.0):
        super().__init__()
        d0 = backbone_dims[0]
        self.input_proj = nn.Linear(in_feats, d0)
        self.ch_embed = nn.Parameter(torch.randn(1, n_channels, d0) * 0.02)
        dims = [d0] + backbone_dims
        self.backbone = nn.ModuleList([
            GATLayer(dims[i], dims[i+1], num_heads, attn_dropout, residual=True)
            for i in range(len(backbone_dims))
        ])
        self.head_aro = TaskHead(backbone_dims[-1], dense_size, head_dropout)
        self.head_val = TaskHead(backbone_dims[-1], dense_size, head_dropout)

    def forward(self, x: torch.Tensor):
        x = F.gelu(self.input_proj(x)) + self.ch_embed
        for layer in self.backbone:
            x, _ = layer(x)
        return torch.cat([self.head_aro(x), self.head_val(x)], dim=1)


# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD MODEL & SCALER
# ─────────────────────────────────────────────────────────────────────────────
print(f"Initializing DeepGAT on {DEVICE}...")
model = DeepGAT(
    n_channels=N_CH, in_feats=N_FEATS,
    backbone_dims=[64, 64, 64], dense_size=128, num_heads=4,
    attn_dropout=0.05, head_dropout=0.3
).to(DEVICE)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print(f"✓ DeepGAT model loaded (91% DEAP).")
else:
    print(f"⚠️ WARNING: Model not found at {MODEL_PATH}")

trained_scaler = None
if os.path.exists(SCALER_PATH):
    with open(SCALER_PATH, 'rb') as f:
        trained_scaler = pickle.load(f)
    print(f"✓ Scaler loaded (160-dim, full feature vector).")
else:
    print(f"⚠️ WARNING: Scaler not found at {SCALER_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────
def differential_entropy(sig, band_edges, fs):
    de = []
    fft_full = np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(len(sig), 1.0 / fs)
    for lo, hi in zip(band_edges[:-1], band_edges[1:]):
        mask = (freqs >= lo) & (freqs < hi)
        f = np.zeros_like(fft_full)
        f[mask] = fft_full[mask]
        var = np.var(np.fft.irfft(f, n=len(sig))) + 1e-10
        de.append(0.5 * np.log(2 * np.pi * np.e * var))
    return de


def extract_features(raw_window_data, sample_rate=128):
    """Extract 10 features per channel (5 BP + 5 DE)."""
    band_edges = [4, 8, 12, 16, 25, 45]
    feats = []
    for ch in range(raw_window_data.shape[0]):
        sig = raw_window_data[ch, :]
        bp = list(pe.bin_power(sig, band_edges, sample_rate)[0])
        de = differential_entropy(sig, band_edges, sample_rate)
        feats.extend(bp + de)
    return np.array(feats, dtype=np.float32)


def preprocess_and_predict(features_vector):
    """Scale features (flattened, 160-dim) and reshape for model input."""
    feat_flat = features_vector.reshape(1, -1)  # (1, 160)
    if trained_scaler is not None:
        feat_flat = trained_scaler.transform(feat_flat)
    feat_3d = feat_flat.reshape(1, N_CH, N_FEATS)  # (1, 16, 10)
    return torch.tensor(feat_3d).float().to(DEVICE)


# ─────────────────────────────────────────────────────────────────────────────
# 5. WEBSOCKET SERVER (Inference Pipeline)
# ─────────────────────────────────────────────────────────────────────────────
prediction_history = deque(maxlen=BUFFER_CAPACITY)
connected_clients = set()


async def broadcast_msg(msg_dict):
    if not connected_clients:
        return
    msg_str = json.dumps(msg_dict)
    tasks = [asyncio.create_task(c.send(msg_str)) for c in connected_clients]
    if tasks:
        await asyncio.wait(tasks)


async def handler(websocket):
    print("-> Client connected!")
    connected_clients.add(websocket)
    data_buffer = np.zeros((N_CH, 0))
    samples_accumulated = 0

    try:
        async for message in websocket:
            try:
                received_data = json.loads(message)

                # Route control commands (forward to other clients)
                if isinstance(received_data, dict):
                    msg_type = received_data.get("type")
                    if msg_type in ["cmd_start_stream", "cmd_pause_stream",
                                    "cmd_resume_stream", "stream_info",
                                    "stream_end", "progress"]:
                        cmd_msg = json.dumps(received_data)
                        tasks = [asyncio.create_task(c.send(cmd_msg))
                                 for c in connected_clients if c != websocket]
                        if tasks:
                            await asyncio.wait(tasks)

                        if msg_type == "cmd_start_stream":
                            data_buffer = np.zeros((N_CH, 0))
                            prediction_history.clear()
                        continue

                # Process raw EEG signal from streamer
                if isinstance(received_data, list):
                    chunk = np.array(received_data)
                    if chunk.shape[1] > 0:
                        await broadcast_msg({"type": "signal", "data": chunk.tolist()})

                        data_buffer = np.concatenate((data_buffer, chunk), axis=1)
                        samples_accumulated += chunk.shape[1]

                        if data_buffer.shape[1] > WINDOW_SIZE:
                            data_buffer = data_buffer[:, -WINDOW_SIZE:]

                        if data_buffer.shape[1] == WINDOW_SIZE and samples_accumulated >= INFERENCE_STEP:
                            samples_accumulated = 0
                            feats = extract_features(data_buffer, sample_rate=128)
                            tens = preprocess_and_predict(feats)

                            with torch.no_grad():
                                out = model(tens)
                                probs = torch.sigmoid(out).cpu().numpy()[0]

                            # NOTE: Model was trained with swapped labels
                            # (DEAP labels[:, 0]=valence, labels[:, 1]=arousal)
                            # but training used all_L[:, 0] as "arousal" and all_L[:, 1] as "valence"
                            # So: head_aro actually predicts valence, head_val actually predicts arousal
                            prob_valence = probs[0]  # head_aro output = actually valence
                            prob_arousal = probs[1]  # head_val output = actually arousal

                            prediction_history.append((prob_arousal, prob_valence))
                            avg_aro = sum(p[0] for p in prediction_history) / len(prediction_history)
                            avg_val = sum(p[1] for p in prediction_history) / len(prediction_history)

                            await broadcast_msg({
                                "type": "prediction",
                                "arousal": float(avg_aro),
                                "valence": float(avg_val)
                            })

            except Exception:
                pass
    except websockets.exceptions.ConnectionClosed:
        print("<- Client disconnected.")
    finally:
        connected_clients.discard(websocket)


# ─────────────────────────────────────────────────────────────────────────────
# 6. HTTP SERVER FOR DASHBOARD
# ─────────────────────────────────────────────────────────────────────────────
class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=".", **kwargs)
    def log_message(self, format, *args):
        pass

def run_http_server():
    with socketserver.TCPServer(("", HTTP_PORT), QuietHandler) as httpd:
        print(f"🌐 HTTP server started: http://localhost:{HTTP_PORT}")
        httpd.serve_forever()


# ─────────────────────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────────────────────
async def main():
    threading.Thread(target=run_http_server, daemon=True).start()
    print(f"📡 WebSocket server started on port {LOCAL_PORT}...")
    async with websockets.serve(handler, "0.0.0.0", LOCAL_PORT, ping_interval=None):
        await asyncio.Future()

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        print("\nStopped.")

PROJECT_ROOT: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork
MODEL_PATH:   c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\work\best_model.pth
SCALER_PATH:  c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\DEAP\output\work\scaler.pkl
Initializing DeepGAT on cuda...
✓ DeepGAT model loaded (91% DEAP).
✓ Scaler loaded (160-dim, full feature vector).
📡 WebSocket server started on port 65432...
🌐 HTTP server started: http://localhost:8000
-> Client connected!
-> Client connected!
-> Client connected!
<- Client disconnected.
